# Delta-Hedged Options P&L Attribution

**Core question:** When we sell an ATM straddle and delta-hedge daily, where does the P&L actually come from?

The Black-Scholes PDE tells us:
$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} = rV$$
$$\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma = rV$$

This means **theta and gamma are two sides of the same coin** — collecting time decay (positive theta) requires being short gamma (exposed to large moves). The net P&L depends on whether implied vol (used to price options) exceeds the realized vol of the underlying.

Net P&L at expiry ≈ $\frac{1}{2} \int_0^T \Gamma S^2 (\sigma_{IV}^2 - \sigma_{RV}^2) \, dt$

---

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from datetime import date

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

TODAY = date.today().isoformat()
TICKER = 'SPY'
print(f"Running as of: {TODAY}")

## 1. Theta-Gamma Trade-off — The Core Mechanism

Before running the backtest, visualize the daily trade-off between theta decay and gamma risk for different volatility regimes.

In [ ]:
from src.pricing.black_scholes import price as bs_price, theta, gamma, vega

# Straddle: sell 1 call + 1 put ATM, 1 contract = 100 shares
S, K, r, q = 450.0, 450.0, 0.05, 0.013
SCALE = 100  # 1 contract

# How does the theta/gamma ratio change as we approach expiry?
days_to_expiry = np.arange(1, 46)
sigmas = [0.15, 0.20, 0.25, 0.30]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = plt.cm.plasma([0.1, 0.35, 0.65, 0.9])

for sigma, color in zip(sigmas, colors):
    theta_vals, gamma_vals, breakeven_vals = [], [], []
    for dte in days_to_expiry:
        T = dte / 365
        th = -(theta(S, K, T, r, sigma, 'call', q) + theta(S, K, T, r, sigma, 'put', q)) * SCALE
        gm = (gamma(S, K, T, r, sigma, q) * 2) * SCALE  # straddle gamma = 2 × single gamma
        # Gamma P&L breakeven move: |dS| where ½ΓdS² = Θ → dS = √(2Θ/Γ)
        be = np.sqrt(2 * th / gm) if gm > 0 else np.nan
        theta_vals.append(th)
        gamma_vals.append(gm)
        breakeven_vals.append(be)

    axes[0].plot(days_to_expiry, theta_vals, color=color, lw=2, label=f'σ={sigma:.0%}')
    axes[1].plot(days_to_expiry, gamma_vals, color=color, lw=2, label=f'σ={sigma:.0%}')
    axes[2].plot(days_to_expiry, breakeven_vals, color=color, lw=2, label=f'σ={sigma:.0%}')

axes[0].set_title('Daily Theta Collected (SHORT straddle, +$)')
axes[0].set_xlabel('Days to Expiry')
axes[0].set_ylabel('$/day')
axes[0].legend(fontsize=9)

axes[1].set_title('Straddle Gamma (risk exposure)')
axes[1].set_xlabel('Days to Expiry')
axes[1].set_ylabel('Gamma per contract')
axes[1].legend(fontsize=9)

axes[2].set_title('Breakeven Daily Move: √(2Θ/Γ)')
axes[2].set_xlabel('Days to Expiry')
axes[2].set_ylabel('$ move that wipes out theta')
axes[2].legend(fontsize=9)

plt.suptitle(f'Short ATM Straddle on SPY — Theta/Gamma Dynamics  (S={S})', fontsize=12)
plt.tight_layout()
plt.show()

print(f"At 30 DTE, σ=20%: theta=${theta_vals[-15]:.1f}/day, breakeven daily move=${breakeven_vals[-15]:.1f}")

## 2. Delta-Hedging Backtest

Run the full simulation: sell 1 ATM straddle on SPY at 30 DTE, hedge delta daily.

In [ ]:
from src.data.fetch import fetch_historical_prices, realized_vol
from src.backtest.delta_hedge import DeltaHedgeBacktest

START = '2023-01-01'
DTE = 30

prices = fetch_historical_prices(TICKER, period='3y')
prices_bt = prices[prices.index >= START].copy()

spot_0 = float(prices_bt['close'].iloc[0])
notional = spot_0 * 100  # 1 contract

print(f"Backtest: {START} → {prices_bt.index[-1].date()}")
print(f"Entry spot: ${spot_0:.2f} | Notional: ${notional:,.0f}")
print(f"Strategy: SHORT 1 ATM straddle, {DTE} DTE, daily delta hedge, $0.01/share TC")

In [ ]:
# Compare two rebalancing frequencies
results = {}
for rebal_freq, label in [(1, 'Daily'), (5, 'Weekly')]:
    bt = DeltaHedgeBacktest(prices_bt, r=0.05, q=0.013, hedge_cost=0.01, rebal_freq=rebal_freq)
    res = bt.run(START, days_to_expiry=DTE, vol_model='rolling_21d')
    results[label] = res
    stats = DeltaHedgeBacktest.summarize(res, notional)
    print(f"\n{label} rebalancing:")
    print(f"  Total P&L:        ${stats['total_pnl']:>8,.2f}")
    print(f"  Annualized Return: {stats['annualized_return']:>7.1%}")
    print(f"  Sharpe Ratio:      {stats['sharpe']:>7.2f}")
    print(f"  Max Drawdown:      {stats['max_drawdown_pct']:>7.1%}")
    print(f"  Win Rate:          {stats['win_rate']:>7.1%}")
    attr = stats['pnl_attribution']
    total = abs(stats['total_pnl'])
    print(f"  Attribution:  Theta={attr['theta']/total:+.0%}  Gamma={attr['gamma']/total:+.0%}  Vega={attr['vega']/total:+.0%}  Costs={attr['transaction']/total:+.0%}")

## 3. Full P&L Attribution Plot

In [ ]:
res = results['Daily']
stats = DeltaHedgeBacktest.summarize(res, notional)

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# --- Cumulative P&L ---
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(res['day'], res['cumulative_pnl'], color='steelblue', lw=2, label='Net P&L')
ax1.fill_between(res['day'], res['cumulative_pnl'], 0,
                 where=res['cumulative_pnl'] >= 0, alpha=0.15, color='green')
ax1.fill_between(res['day'], res['cumulative_pnl'], 0,
                 where=res['cumulative_pnl'] < 0, alpha=0.15, color='red')
ax1.axhline(0, color='black', lw=0.8, ls='--')
ax1.set_title(f'Cumulative P&L — Short ATM Straddle + Daily Delta Hedge  [{START} →]')
ax1.set_xlabel('Days')
ax1.set_ylabel('P&L ($)')
ax1.legend()

# --- Greek attribution ---
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(res['day'], res['cumulative_theta'], color='seagreen', lw=2, label='Theta (collect)')
ax2.plot(res['day'], res['cumulative_gamma'], color='crimson', lw=2, label='Gamma (cost)')
ax2.plot(res['day'], res['cumulative_vega'], color='purple', lw=1.5, label='Vega (vol moves)')
ax2.plot(res['day'], res['cumulative_transaction'], color='brown', lw=1, ls='--', label='TC')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Cumulative P&L by Greek')
ax2.set_xlabel('Days')
ax2.set_ylabel('P&L ($)')
ax2.legend(fontsize=8)

# --- Daily P&L distribution ---
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(res['pnl_total'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax3.axvline(res['pnl_total'].mean(), color='red', lw=2, ls='--',
            label=f"Mean: ${res['pnl_total'].mean():.1f}")
ax3.set_title('Daily P&L Distribution')
ax3.set_xlabel('P&L ($)')
ax3.legend(fontsize=9)

# --- Attribution pie ---
ax4 = fig.add_subplot(gs[1, 1])
attr = stats['pnl_attribution']
pie_labels = ['Theta', 'Gamma', 'Vega', 'Hedge', 'TC']
pie_vals = [abs(attr['theta']), abs(attr['gamma']), abs(attr['vega']),
            abs(attr['hedge']), abs(attr['transaction'])]
pie_colors = ['#2ca02c', '#d62728', '#9467bd', '#1f77b4', '#8c564b']
ax4.pie(pie_vals, labels=pie_labels, colors=pie_colors, autopct='%1.0f%%', startangle=90)
ax4.set_title('P&L Attribution (|absolute|)')

# --- IV vs RV over the period ---
ax5 = fig.add_subplot(gs[1, 2])
ax5.scatter(res['day'], res['iv'] * 100, s=12, alpha=0.6, color='darkorange', label='IV (for pricing)')
daily_rv = res['dS'].abs() / res['spot'] * np.sqrt(252) * 100
ax5.scatter(res['day'], daily_rv, s=12, alpha=0.4, color='steelblue', label='Daily RV (ann.)')
ax5.set_title('IV vs Realized Vol')
ax5.set_xlabel('Days')
ax5.set_ylabel('Vol (%)')
ax5.legend(fontsize=9)

plt.suptitle(
    f"{TICKER} Short Straddle | DTE={DTE} | Sharpe={stats['sharpe']:.2f} | "
    f"Ret={stats['annualized_return']:.1%} ann | MaxDD={stats['max_drawdown_pct']:.1%}",
    fontsize=12, y=1.01
)
plt.show()

## 4. Rebalancing Frequency — Impact of Transaction Costs

More frequent hedging keeps us closer to delta-neutral (reduces unhedged delta exposure) but accumulates more transaction costs. The optimal frequency depends on the gamma/spread ratio.

In [ ]:
rebal_freqs = [1, 2, 3, 5, 10]
summary_rows = []

for freq in rebal_freqs:
    bt = DeltaHedgeBacktest(prices_bt, r=0.05, q=0.013, hedge_cost=0.01, rebal_freq=freq)
    res = bt.run(START, days_to_expiry=DTE, vol_model='rolling_21d')
    stats = DeltaHedgeBacktest.summarize(res, notional)
    summary_rows.append({
        'rebal_freq': freq,
        'total_pnl': stats['total_pnl'],
        'ann_return': stats['annualized_return'],
        'sharpe': stats['sharpe'],
        'max_dd': stats['max_drawdown_pct'],
        'win_rate': stats['win_rate'],
        'tc': stats['pnl_attribution']['transaction'],
    })

summary_df = pd.DataFrame(summary_rows)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(summary_df['rebal_freq'], summary_df['sharpe'], 'o-', color='steelblue', lw=2, ms=8)
axes[0].set_title('Sharpe Ratio vs Rebalancing Frequency')
axes[0].set_xlabel('Rebalance Every N Days')
axes[0].set_ylabel('Sharpe')

axes[1].plot(summary_df['rebal_freq'], summary_df['ann_return'] * 100, 'o-', color='seagreen', lw=2, ms=8)
axes[1].set_title('Annualized Return vs Rebalancing Frequency')
axes[1].set_xlabel('Rebalance Every N Days')
axes[1].set_ylabel('Annualized Return (%)')

axes[2].bar(summary_df['rebal_freq'], summary_df['tc'].abs(), color='crimson', alpha=0.7)
axes[2].set_title('Total Transaction Costs vs Rebalancing Frequency')
axes[2].set_xlabel('Rebalance Every N Days')
axes[2].set_ylabel('TC ($)')

plt.tight_layout()
plt.show()

print("\nFull summary:")
display(summary_df.round(3))

## 5. Sensitivity to Entry Vol Level (VIX Regime)

The strategy's profitability depends critically on the IV at entry. High entry IV (high VIX) means more premium collected but also more realized volatility risk. Let's simulate entry at different IV levels.

In [ ]:
entry_ivs = [0.12, 0.15, 0.20, 0.25, 0.30, 0.40]
iv_rows = []

for iv_entry in entry_ivs:
    bt = DeltaHedgeBacktest(prices_bt, r=0.05, q=0.013, hedge_cost=0.01, rebal_freq=1)
    res = bt.run(START, days_to_expiry=DTE, initial_iv=iv_entry, vol_model='flat')
    stats = DeltaHedgeBacktest.summarize(res, notional)
    iv_rows.append({
        'entry_iv': iv_entry,
        'total_pnl': stats['total_pnl'],
        'ann_return': stats['annualized_return'],
        'sharpe': stats['sharpe'],
        'theta_total': stats['pnl_attribution']['theta'],
        'gamma_total': stats['pnl_attribution']['gamma'],
    })

iv_df = pd.DataFrame(iv_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(iv_df['entry_iv'] * 100, iv_df['ann_return'] * 100, 'o-', color='steelblue', lw=2, ms=8, label='Ann. Return')
axes[0].axhline(0, color='black', lw=0.8, ls='--')
axes[0].set_title('Annualized Return vs Entry IV')
axes[0].set_xlabel('Entry Implied Vol (%)')
axes[0].set_ylabel('Return (%)')

axes[1].plot(iv_df['entry_iv'] * 100, iv_df['theta_total'], 'o-', color='seagreen', lw=2, ms=8, label='Theta collected')
axes[1].plot(iv_df['entry_iv'] * 100, iv_df['gamma_total'], 's-', color='crimson', lw=2, ms=8, label='Gamma cost')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Theta vs Gamma P&L by Entry IV')
axes[1].set_xlabel('Entry Implied Vol (%)')
axes[1].set_ylabel('P&L ($)')
axes[1].legend(fontsize=9)

plt.suptitle('Sensitivity to Entry IV Level  (flat vol model, no vol regime adjustment)', fontsize=12)
plt.tight_layout()
plt.show()

display(iv_df.round(3))

### Interpretation

- **Higher entry IV → more premium collected (theta) but also more gamma risk**
- The optimal regime for this strategy is **moderate VIX (15–25)**: enough premium to justify the risk, but not so high that realized vol blows through the premium collected
- At very high IV (VIX > 35), the strategy often loses because large market moves (high gamma cost) exceed the premium collected

This motivates a **regime-conditional sizing rule**: scale position size by `1 / VIX_level`, collecting more premium per unit of risk in calm markets.

---

### Next steps
- Notebook 03: Extend to a regime-conditioned strategy with VIX-based position sizing
- Notebook 04: Apply the surface to price exotic structures (calendar spreads, iron condors)